# European Crime & Migration — Time Series

Data source: **Eurostat** — Statistics JSON API (open access, no key required)  

- **Top plot**: Immigration volume by country over time  
- **Bottom plot**: Intentional homicide rate (per 100 000 inhabitants) by country over time  

Countries: major EU member states with available data.

In [2]:
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

pio.renderers.default = "vscode" if os.environ.get("VSCODE_PID") else "notebook_connected"

# ══════════════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════════════
EUROSTAT_BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"

COUNTRIES = [
    "DE", "FR", "IT", "ES", "NL", "BE", "SE", "AT", "PL", "PT",
    "CZ", "DK", "FI", "IE", "EL", "RO", "HU", "BG", "HR"
]

# Muted colour palette — one colour per country
PALETTE = [
    "#4F7F84",  # DE  Blue-green
    "#C4787A",  # FR  Dusty red
    "#494C30",  # IT  Olive
    "#D4A574",  # ES  Sand
    "#7A9E9F",  # NL  Teal
    "#B8860B",  # BE  Dark goldenrod
    "#8FBC8F",  # SE  Dark sea green
    "#A0522D",  # AT  Sienna
    "#9B7CB8",  # PL  Soft purple
    "#CD853F",  # PT  Peru
    "#6B8E9B",  # CZ  Steel blue
    "#BC8F8F",  # DK  Rosy brown
    "#5F9EA0",  # FI  Cadet blue
    "#D2B48C",  # IE  Tan
    "#708090",  # EL  Slate grey
    "#BDB76B",  # RO  Dark khaki
    "#8B4513",  # HU  Saddle brown
    "#778899",  # BG  Light slate grey
    "#DAA520",  # HR  Goldenrod
]
COLOR_MAP = dict(zip(COUNTRIES, PALETTE))

# ── Theme constants (matching IQ notebook) ───────────────────────────
BG_COLOR      = "#EEE5E0"
PAPER_COLOR   = "#1E1B16"
TEXT_COLOR    = "#FFE8C8"
TITLE_COLOR   = "#FFF5E6"
GRID_COLOR    = "rgba(255, 235, 200, 0.08)"
LINE_COLOR    = "rgba(255, 235, 200, 0.25)"


# ══════════════════════════════════════════════════════════════════════
# Helper: decode Eurostat JSON-stat into a DataFrame
# ══════════════════════════════════════════════════════════════════════
def eurostat_to_df(data, value_col):
    """Flatten a Eurostat JSON-stat response to a DataFrame.

    Handles the multi-dimensional index math automatically by
    iterating over all non-singleton dimensions (geo × time).
    """
    dims = data["id"]                     # ordered dimension names
    sizes = data["size"]                   # sizes of each dim
    geo_labels = data["dimension"]["geo"]["category"]["label"]
    geo_index  = data["dimension"]["geo"]["category"]["index"]
    time_index = data["dimension"]["time"]["category"]["index"]

    # Find strides for geo & time dims
    geo_dim_pos  = dims.index("geo")
    time_dim_pos = dims.index("time")

    # Compute flat index from dimension positions
    def flat_index(dim_indices):
        idx = 0
        for i, di in enumerate(dim_indices):
            stride = 1
            for j in range(i + 1, len(sizes)):
                stride *= sizes[j]
            idx += di * stride
        return idx

    # Build a base vector of zeros (all dims at index 0)
    base = [0] * len(dims)

    rows = []
    for geo_code, gi in geo_index.items():
        for year, ti in time_index.items():
            vec = base.copy()
            vec[geo_dim_pos] = gi
            vec[time_dim_pos] = ti
            pos = str(flat_index(vec))
            val = data["value"].get(pos)
            if val is not None:
                rows.append({"country": geo_labels[geo_code],
                             "iso2": geo_code,
                             "year": int(year),
                             value_col: val})
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════
# Fetch data
# ══════════════════════════════════════════════════════════════════════
print("Fetching immigration data …")
migr_resp = requests.get(
    f"{EUROSTAT_BASE}/migr_imm1ctz",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "citizen": "TOTAL", "age": "TOTAL",
        "sex": "T", "agedef": "REACH",
        "geo": COUNTRIES,
    },
    timeout=30,
)
migr_resp.raise_for_status()
df_migr = eurostat_to_df(migr_resp.json(), "immigrants")
df_migr["immigrants_m"] = df_migr["immigrants"] / 1_000_000   # to millions
print(f"  ✓ {len(df_migr)} rows  ({df_migr.year.min()}–{df_migr.year.max()})")

print("Fetching crime data …")
crime_resp = requests.get(
    f"{EUROSTAT_BASE}/crim_off_cat",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "iccs": "ICCS0101",
        "unit": "P_HTHAB",
        "geo": COUNTRIES,
    },
    timeout=30,
)
crime_resp.raise_for_status()
df_crime = eurostat_to_df(crime_resp.json(), "homicide_rate")
print(f"  ✓ {len(df_crime)} rows  ({df_crime.year.min()}–{df_crime.year.max()})")


# ══════════════════════════════════════════════════════════════════════
# Build subplots: top = immigration, bottom = crime
# ══════════════════════════════════════════════════════════════════════
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=["Immigration (total arrivals per year)",
                    "Intentional Homicide Rate (per 100 000 inhabitants)"],
)

# ── Top plot: immigration time series ─────────────────────────────────
for iso2 in COUNTRIES:
    sub = df_migr[df_migr.iso2 == iso2].sort_values("year")
    if sub.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=sub["year"], y=sub["immigrants_m"],
            mode="lines+markers",
            name=sub["country"].iloc[0],
            line=dict(color=COLOR_MAP[iso2], width=2),
            marker=dict(size=4),
            hovertemplate="<b>%{fullData.name}</b><br>Year: %{x}<br>Immigrants: %{y:.2f}M<extra></extra>",
        ),
        row=1, col=1,
    )

# ── Bottom plot: crime rate time series ───────────────────────────────
for iso2 in COUNTRIES:
    sub = df_crime[df_crime.iso2 == iso2].sort_values("year")
    if sub.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=sub["year"], y=sub["homicide_rate"],
            mode="lines+markers",
            name=sub["country"].iloc[0],
            line=dict(color=COLOR_MAP[iso2], width=2),
            marker=dict(size=4),
            showlegend=False,
            hovertemplate="<b>%{fullData.name}</b><br>Year: %{x}<br>Rate: %{y:.2f} /100k<extra></extra>",
        ),
        row=2, col=1,
    )

# ── Layout ────────────────────────────────────────────────────────────
fig.update_layout(
    height=900,
    width=1100,
    paper_bgcolor=PAPER_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR),
    title=dict(
        text="<b>European Immigration & Crime — Time Series</b>"
             '<br><span style="font-size:13px; color:rgba(255,232,200,0.55)">'
             'Source: Eurostat  ·  Major EU member states  ·  2010 – latest</span>',
        font=dict(color=TITLE_COLOR, size=22,
                  family="Inter, Segoe UI, sans-serif"),
        x=0.5, xanchor='center',
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,235,200,0.15)",
        borderwidth=1,
        font=dict(color=TEXT_COLOR, size=11,
                  family="Inter, sans-serif"),
        x=1.02, y=1.0, xanchor='left', yanchor='top',
    ),
    hovermode='x unified',
    margin=dict(l=70, r=160, t=100, b=50),
)

# Style all axes
for ax in ['xaxis', 'xaxis2']:
    fig.update_layout(**{ax: dict(
        gridcolor=GRID_COLOR,
        linecolor=LINE_COLOR,
        zerolinecolor=LINE_COLOR,
        showgrid=True,
        tickfont=dict(color=TEXT_COLOR, family='JetBrains Mono, monospace', size=12),
        dtick=1,
    )})

for i, (ax, lbl) in enumerate([('yaxis', 'Immigrants (millions)'),
                                ('yaxis2', 'Homicides per 100k')]):
    fig.update_layout(**{ax: dict(
        gridcolor=GRID_COLOR,
        linecolor=LINE_COLOR,
        zerolinecolor=LINE_COLOR,
        showgrid=True,
        tickfont=dict(color=TEXT_COLOR, family='JetBrains Mono, monospace', size=12),
        title=dict(text=lbl, font=dict(color=TEXT_COLOR, size=14)),
    )})

# Style subplot titles
for ann in fig.layout.annotations:
    ann.font = dict(color=TEXT_COLOR, size=14, family='Inter, sans-serif')

# ── Export ────────────────────────────────────────────────────────────
os.makedirs("output", exist_ok=True)
fig.write_html("output/europe_crime_migration.html")
fig.write_image("output/europe_crime_migration.png", scale=2)
print("Saved → output/europe_crime_migration.html  &  output/europe_crime_migration.png")

fig.show()


Fetching immigration data …
  ✓ 277 rows  (2010–2024)
Fetching crime data …
  ✓ 263 rows  (2010–2023)
Saved → output/europe_crime_migration.html  &  output/europe_crime_migration.png


In [3]:
# ══════════════════════════════════════════════════════════════════════
# 2×2 Country Deep-Dive
#   Stacked area  = Native population (dark) + Foreign-citizen stock (light)
#   Crime lines   = Total homicides (bright)  +  Serious assault (darker)
# ══════════════════════════════════════════════════════════════════════
from plotly.subplots import make_subplots
import plotly.graph_objects as go

FOCUS = {
    "DE": {"name": "Germany",  "base": "#4F7F84"},
    "FR": {"name": "France",   "base": "#C4787A"},
    "ES": {"name": "Spain",    "base": "#D4A574"},
    "IT": {"name": "Italy",    "base": "#494C30"},
}
focus_codes = list(FOCUS.keys())

# ── Colour helpers ────────────────────────────────────────────────────
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def shade(h, factor, alpha=1.0):
    """Lighten (>1) or darken (<1) a hex colour, with optional alpha."""
    r, g, b = hex_to_rgb(h)
    r = min(255, int(r * factor))
    g = min(255, int(g * factor))
    b = min(255, int(b * factor))
    if alpha < 1.0:
        return f"rgba({r},{g},{b},{alpha})"
    return f"rgb({r},{g},{b})"


# ══════════════════════════════════════════════════════════════════════
# Fetch population by citizenship: TOTAL + NAT → foreign = TOTAL - NAT
# ══════════════════════════════════════════════════════════════════════
def eurostat_to_df_ctz(data, value_col):
    """Decode migr_pop1ctz JSON-stat with citizen dimension."""
    dims   = data["id"]
    sizes  = data["size"]
    geo_labels = data["dimension"]["geo"]["category"]["label"]
    geo_index  = data["dimension"]["geo"]["category"]["index"]
    time_index = data["dimension"]["time"]["category"]["index"]
    ctz_index  = data["dimension"]["citizen"]["category"]["index"]
    ctz_labels = data["dimension"]["citizen"]["category"]["label"]

    def pos_of(dim_name): return dims.index(dim_name)

    def flat_index(vec):
        idx = 0
        for i, di in enumerate(vec):
            stride = 1
            for j in range(i + 1, len(sizes)):
                stride *= sizes[j]
            idx += di * stride
        return idx

    base = [0] * len(dims)
    rows = []
    for ctz_code, ci in ctz_index.items():
        for geo_code, gi in geo_index.items():
            for year, ti in time_index.items():
                vec = base.copy()
                vec[pos_of("citizen")] = ci
                vec[pos_of("geo")]     = gi
                vec[pos_of("time")]    = ti
                val = data["value"].get(str(flat_index(vec)))
                if val is not None:
                    rows.append({
                        "citizen": ctz_code,
                        "country": geo_labels[geo_code],
                        "iso2": geo_code,
                        "year": int(year),
                        value_col: val,
                    })
    return pd.DataFrame(rows)


print("Fetching population by citizenship (TOTAL + NAT) …")
pop_ctz_resp = requests.get(
    f"{EUROSTAT_BASE}/migr_pop1ctz",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "age": "TOTAL", "sex": "T",
        "citizen": ["TOTAL", "NAT"],
        "geo": focus_codes,
    },
    timeout=30,
)
pop_ctz_resp.raise_for_status()
df_ctz = eurostat_to_df_ctz(pop_ctz_resp.json(), "population")
df_ctz["pop_m"] = df_ctz["population"] / 1_000_000

# Pivot: one row per (iso2, year) with columns pop_total_m, pop_nat_m
df_tot = df_ctz[df_ctz.citizen == "TOTAL"][["iso2", "country", "year", "pop_m"]].rename(columns={"pop_m": "pop_total_m"})
df_nat = df_ctz[df_ctz.citizen == "NAT"][["iso2", "year", "pop_m"]].rename(columns={"pop_m": "pop_nat_m"})
df_pop2 = df_tot.merge(df_nat, on=["iso2", "year"], how="inner")
df_pop2["pop_foreign_m"] = df_pop2["pop_total_m"] - df_pop2["pop_nat_m"]
print(f"  ✓ {len(df_pop2)} rows  ({df_pop2.year.min()}–{df_pop2.year.max()})")


# ══════════════════════════════════════════════════════════════════════
# Fetch crime: total homicides (ICCS0101) + serious assault (ICCS020111)
# ══════════════════════════════════════════════════════════════════════
def eurostat_to_df_crime(data, value_col):
    """Decode crim_off_cat JSON-stat with iccs dimension."""
    dims   = data["id"]
    sizes  = data["size"]
    geo_labels = data["dimension"]["geo"]["category"]["label"]
    geo_index  = data["dimension"]["geo"]["category"]["index"]
    time_index = data["dimension"]["time"]["category"]["index"]
    iccs_index = data["dimension"]["iccs"]["category"]["index"]
    iccs_labels = data["dimension"]["iccs"]["category"]["label"]

    def pos_of(dim_name): return dims.index(dim_name)

    def flat_index(vec):
        idx = 0
        for i, di in enumerate(vec):
            stride = 1
            for j in range(i + 1, len(sizes)):
                stride *= sizes[j]
            idx += di * stride
        return idx

    base = [0] * len(dims)
    rows = []
    for iccs_code, ii in iccs_index.items():
        for geo_code, gi in geo_index.items():
            for year, ti in time_index.items():
                vec = base.copy()
                vec[pos_of("iccs")] = ii
                vec[pos_of("geo")]  = gi
                vec[pos_of("time")] = ti
                val = data["value"].get(str(flat_index(vec)))
                if val is not None:
                    rows.append({
                        "iccs": iccs_code,
                        "iccs_label": iccs_labels[iccs_code],
                        "country": geo_labels[geo_code],
                        "iso2": geo_code,
                        "year": int(year),
                        value_col: val,
                    })
    return pd.DataFrame(rows)


print("Fetching crime totals (homicides + serious assault) …")
crime2_resp = requests.get(
    f"{EUROSTAT_BASE}/crim_off_cat",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "iccs": ["ICCS0101", "ICCS020111"],
        "unit": "NR",
        "geo": focus_codes,
    },
    timeout=30,
)
crime2_resp.raise_for_status()
df_crime2 = eurostat_to_df_crime(crime2_resp.json(), "offences")
print(f"  ✓ {len(df_crime2)} rows  ({df_crime2.year.min()}–{df_crime2.year.max()})")

# Split into two DataFrames
df_homicide = df_crime2[df_crime2.iccs == "ICCS0101"].copy()
df_assault  = df_crime2[df_crime2.iccs == "ICCS020111"].copy()


# ══════════════════════════════════════════════════════════════════════
# Build 2×2 subplot grid (secondary_y on each)
# ══════════════════════════════════════════════════════════════════════
fig2 = make_subplots(
    rows=2, cols=2,
    shared_xaxes=True,
    vertical_spacing=0.10,
    horizontal_spacing=0.12,
    subplot_titles=[FOCUS[c]["name"] for c in focus_codes],
    specs=[[{"secondary_y": True}, {"secondary_y": True}],
           [{"secondary_y": True}, {"secondary_y": True}]],
)

for idx, iso2 in enumerate(focus_codes):
    row = idx // 2 + 1
    col = idx % 2 + 1
    base_hex = FOCUS[iso2]["base"]
    dark  = shade(base_hex, 0.65)       # darker  — native pop
    light = shade(base_hex, 1.50, 0.7)  # lighter — foreign pop

    pop_s  = df_pop2[df_pop2.iso2 == iso2].sort_values("year")
    hom_s  = df_homicide[df_homicide.iso2 == iso2].sort_values("year")
    ass_s  = df_assault[df_assault.iso2 == iso2].sort_values("year")

    # ── Stacked area: native (bottom, dark) ──────────────────────────
    fig2.add_trace(
        go.Scatter(
            x=pop_s["year"], y=pop_s["pop_nat_m"],
            mode="lines", name="Native population",
            line=dict(width=0), fillcolor=dark,
            fill="tozeroy", showlegend=(idx == 0),
            legendgroup="native",
            hovertemplate="Native: %{y:.1f}M<extra></extra>",
        ),
        row=row, col=col, secondary_y=False,
    )

    # ── Stacked area: foreign on top (lighter shade) ─────────────────
    fig2.add_trace(
        go.Scatter(
            x=pop_s["year"], y=pop_s["pop_total_m"],
            mode="lines", name="Foreign population",
            line=dict(width=0), fillcolor=light,
            fill="tonexty", showlegend=(idx == 0),
            legendgroup="foreign",
            hovertemplate="Total pop: %{y:.1f}M  (foreign: %{customdata:.1f}M)<extra></extra>",
            customdata=pop_s["pop_foreign_m"],
        ),
        row=row, col=col, secondary_y=False,
    )

    # ── Crime line 1: serious assault (darker red) ───────────────────
    fig2.add_trace(
        go.Scatter(
            x=ass_s["year"], y=ass_s["offences"],
            mode="lines+markers", name="Serious assault",
            line=dict(color="#B22222", width=2.5),
            marker=dict(size=4, color="#B22222"),
            showlegend=(idx == 0),
            legendgroup="assault",
            hovertemplate="Assault: %{y:,.0f}<extra></extra>",
        ),
        row=row, col=col, secondary_y=True,
    )

    # ── Crime line 2: intentional homicide (bright red) ──────────────
    fig2.add_trace(
        go.Scatter(
            x=hom_s["year"], y=hom_s["offences"],
            mode="lines+markers", name="Intentional homicide",
            line=dict(color="#FF6B6B", width=2, dash="dot"),
            marker=dict(size=5, color="#FF6B6B", symbol="diamond"),
            showlegend=(idx == 0),
            legendgroup="homicide",
            hovertemplate="Homicides: %{y:,.0f}<extra></extra>",
        ),
        row=row, col=col, secondary_y=True,
    )


# ── Layout ────────────────────────────────────────────────────────────
fig2.update_layout(
    height=850,
    width=1150,
    paper_bgcolor=PAPER_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR),
    title=dict(
        text="<b>Population Composition & Violent Crime — Country Profiles</b>"
             '<br><span style="font-size:13px; color:rgba(255,232,200,0.55)">'
             "Source: Eurostat  ·  Stacked area = native (dark) + foreign citizens (light)"
             "  ·  Lines = serious assault (dark red) / homicides (bright red, dotted)</span>",
        font=dict(color=TITLE_COLOR, size=19,
                  family="Inter, Segoe UI, sans-serif"),
        x=0.5, xanchor="center",
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,235,200,0.15)",
        borderwidth=1,
        font=dict(color=TEXT_COLOR, size=11,
                  family="Inter, sans-serif"),
        orientation="h",
        x=0.5, y=-0.06, xanchor="center",
    ),
    hovermode="x unified",
    margin=dict(l=65, r=65, t=110, b=70),
)

# Style x-axes
for i in range(1, 5):
    xa = f"xaxis{i}" if i > 1 else "xaxis"
    fig2.update_layout(**{xa: dict(
        gridcolor=GRID_COLOR, linecolor=LINE_COLOR,
        tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=11),
        dtick=2,
    )})

# Primary y (population)
fig2.update_yaxes(
    gridcolor=GRID_COLOR, linecolor=LINE_COLOR,
    tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=11),
    ticksuffix="M",
    secondary_y=False,
)

# Secondary y (crime counts)
fig2.update_yaxes(
    gridcolor="rgba(255, 100, 100, 0.12)",
    linecolor="rgba(255, 100, 100, 0.25)",
    tickfont=dict(color="#FF6B6B", family="JetBrains Mono, monospace", size=10),
    secondary_y=True,
)

# Style subplot titles
for ann in fig2.layout.annotations:
    ann.font = dict(color=TEXT_COLOR, size=14, family="Inter, sans-serif")

# ── Export ────────────────────────────────────────────────────────────
fig2.write_html("output/europe_country_profiles.html")
fig2.write_image("output/europe_country_profiles.png", scale=2)
print("Saved → output/europe_country_profiles.html  &  .png")

fig2.show()


Fetching population by citizenship (TOTAL + NAT) …
  ✓ 64 rows  (2010–2025)
Fetching crime totals (homicides + serious assault) …
  ✓ 112 rows  (2010–2023)
Saved → output/europe_country_profiles.html  &  .png


In [4]:
# ══════════════════════════════════════════════════════════════════════
# 2×2 Country Deep-Dive: Migrant Population + Non-Employment + Welfare
#   Filled area = Foreign-citizen population (millions)
#   Solid line  = Non-employment rate of non-EU citizens (100% − emp rate)
#   Dotted line = At-risk-of-poverty rate of foreign citizens (%)
#                 = % of immigrant pop at risk of poverty / receiving welfare
# ══════════════════════════════════════════════════════════════════════
from plotly.subplots import make_subplots
import plotly.graph_objects as go

FOCUS = {
    "DE": {"name": "Germany",  "color": "#EEB5A4"},  # Rose Quartz
    "FR": {"name": "France",   "color": "#88232B"},  # Crimson
    "ES": {"name": "Spain",    "color": "#9FA975"},  # Lint
    "IT": {"name": "Italy",    "color": "#7C756B"},  # Cool Gray
}
focus_codes = list(FOCUS.keys())


# ── Colour helpers ────────────────────────────────────────────────────
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def rgba(h, alpha=1.0):
    r, g, b = hex_to_rgb(h)
    return f"rgba({r},{g},{b},{alpha})"

def shade(h, factor, alpha=1.0):
    r, g, b = hex_to_rgb(h)
    r = min(255, int(r * factor))
    g = min(255, int(g * factor))
    b = min(255, int(b * factor))
    if alpha < 1.0:
        return f"rgba({r},{g},{b},{alpha})"
    return f"rgb({r},{g},{b})"


# ══════════════════════════════════════════════════════════════════════
# Generic decoder
# ══════════════════════════════════════════════════════════════════════
def eurostat_flat(data, extra_dim=None):
    dims  = data["id"]
    sizes = data["size"]
    geo_labels = data["dimension"]["geo"]["category"]["label"]
    geo_index  = data["dimension"]["geo"]["category"]["index"]
    time_index = data["dimension"]["time"]["category"]["index"]

    extra_index, extra_labels = {}, {}
    if extra_dim:
        extra_index = data["dimension"][extra_dim]["category"]["index"]
        extra_labels = data["dimension"][extra_dim]["category"]["label"]

    def flat_idx(vec):
        idx = 0
        for i, di in enumerate(vec):
            s = 1
            for j in range(i + 1, len(sizes)):
                s *= sizes[j]
            idx += di * s
        return idx

    base = [0] * len(dims)
    rows = []
    ex_items = extra_index.items() if extra_dim else [("_", 0)]
    for ex_code, ei in ex_items:
        for geo_code, gi in geo_index.items():
            for year, ti in time_index.items():
                vec = base.copy()
                vec[dims.index("geo")]  = gi
                vec[dims.index("time")] = ti
                if extra_dim:
                    vec[dims.index(extra_dim)] = ei
                val = data["value"].get(str(flat_idx(vec)))
                if val is not None:
                    row = {"country": geo_labels[geo_code],
                           "iso2": geo_code, "year": int(year), "value": val}
                    if extra_dim:
                        row["dim_code"]  = ex_code
                        row["dim_label"] = extra_labels.get(ex_code, ex_code)
                    rows.append(row)
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════
# 1. Foreign population stock (TOTAL − NAT)
# ══════════════════════════════════════════════════════════════════════
print("Fetching population by citizenship …")
pop_resp = requests.get(
    f"{EUROSTAT_BASE}/migr_pop1ctz",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "age": "TOTAL", "sex": "T",
        "citizen": ["TOTAL", "NAT"],
        "geo": focus_codes,
    },
    timeout=30,
)
pop_resp.raise_for_status()
df_pop_raw = eurostat_flat(pop_resp.json(), extra_dim="citizen")

df_tot = (df_pop_raw[df_pop_raw.dim_code == "TOTAL"]
          [["iso2", "country", "year", "value"]]
          .rename(columns={"value": "pop_total"}))
df_nat = (df_pop_raw[df_pop_raw.dim_code == "NAT"]
          [["iso2", "year", "value"]]
          .rename(columns={"value": "pop_nat"}))
df_pop3 = df_tot.merge(df_nat, on=["iso2", "year"], how="inner")
df_pop3["pop_foreign_m"] = (df_pop3["pop_total"] - df_pop3["pop_nat"]) / 1_000_000
print(f"  ✓ {len(df_pop3)} rows  ({df_pop3.year.min()}–{df_pop3.year.max()})")


# ══════════════════════════════════════════════════════════════════════
# 2. Non-employment rate = 100% − employment rate (non-EU citizens, 20-64)
# ══════════════════════════════════════════════════════════════════════
print("Fetching non-EU employment rates …")
emp_resp = requests.get(
    f"{EUROSTAT_BASE}/lfsa_ergan",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "sex": "T", "age": "Y20-64",
        "citizen": "NEU27_2020_FOR",
        "geo": focus_codes,
    },
    timeout=30,
)
emp_resp.raise_for_status()
df_emp = eurostat_flat(emp_resp.json())
df_emp = df_emp.rename(columns={"value": "emp_pct"})
df_emp["nonemp_pct"] = 100.0 - df_emp["emp_pct"]
print(f"  ✓ {len(df_emp)} rows  ({df_emp.year.min()}–{df_emp.year.max()})")


# ══════════════════════════════════════════════════════════════════════
# 3. At-risk-of-poverty rate for foreign citizens (% of immigrant pop)
#    Dataset: ilc_li31 — by citizenship, population aged 18+
# ══════════════════════════════════════════════════════════════════════
print("Fetching at-risk-of-poverty rate for foreign citizens …")
pov_resp = requests.get(
    f"{EUROSTAT_BASE}/ilc_li31",
    params={
        "format": "JSON", "lang": "en",
        "sinceTimePeriod": "2010",
        "sex": "T", "age": "Y_GE18",
        "citizen": "FOR",
        "geo": focus_codes,
    },
    timeout=30,
)
pov_resp.raise_for_status()
df_pov = eurostat_flat(pov_resp.json())
df_pov = df_pov.rename(columns={"value": "poverty_pct"})
print(f"  ✓ {len(df_pov)} rows  ({df_pov.year.min()}–{df_pov.year.max()})")


# ══════════════════════════════════════════════════════════════════════
# Build 2×2 subplots
# ══════════════════════════════════════════════════════════════════════
fig3 = make_subplots(
    rows=2, cols=2,
    shared_xaxes=True,
    vertical_spacing=0.10,
    horizontal_spacing=0.12,
    subplot_titles=[FOCUS[c]["name"] for c in focus_codes],
    specs=[[{"secondary_y": True}, {"secondary_y": True}],
           [{"secondary_y": True}, {"secondary_y": True}]],
)

for idx, iso2 in enumerate(focus_codes):
    row = idx // 2 + 1
    col = idx % 2 + 1
    base_hex = FOCUS[iso2]["color"]

    pop_s = df_pop3[df_pop3.iso2 == iso2].sort_values("year")
    emp_s = df_emp[df_emp.iso2 == iso2].sort_values("year")
    pov_s = df_pov[df_pov.iso2 == iso2].sort_values("year")

    # ── Filled area: foreign population ──────────────────────────────
    fig3.add_trace(
        go.Scatter(
            x=pop_s["year"], y=pop_s["pop_foreign_m"],
            mode="lines",
            name="Foreign population",
            line=dict(width=1, color=rgba(base_hex, 0.8)),
            fillcolor=rgba(base_hex, 0.45),
            fill="tozeroy",
            showlegend=(idx == 0),
            legendgroup="foreign",
            hovertemplate="Foreign pop: %{y:.2f}M<extra></extra>",
        ),
        row=row, col=col, secondary_y=False,
    )

    # ── Solid line: non-employment rate (darker shade) ───────────────
    if not emp_s.empty:
        fig3.add_trace(
            go.Scatter(
                x=emp_s["year"], y=emp_s["nonemp_pct"],
                mode="lines+markers",
                name="Non-employment rate (non-EU)",
                line=dict(color=shade(base_hex, 0.55), width=2.5),
                marker=dict(size=5, color=shade(base_hex, 0.55)),
                showlegend=(idx == 0),
                legendgroup="nonemp",
                hovertemplate="Non-employed: %{y:.1f}%<extra></extra>",
            ),
            row=row, col=col, secondary_y=True,
        )

    # ── Dotted line: poverty rate among immigrants (darkest shade) ────
    if not pov_s.empty:
        fig3.add_trace(
            go.Scatter(
                x=pov_s["year"], y=pov_s["poverty_pct"],
                mode="lines+markers",
                name="Immigrant poverty rate",
                line=dict(color=shade(base_hex, 0.35), width=2, dash="dot"),
                marker=dict(size=5, color=shade(base_hex, 0.35), symbol="diamond"),
                showlegend=(idx == 0),
                legendgroup="poverty",
                hovertemplate="Poverty rate: %{y:.1f}%<extra></extra>",
            ),
            row=row, col=col, secondary_y=True,
        )


# ── Layout ────────────────────────────────────────────────────────────
fig3.update_layout(
    height=850,
    width=1150,
    paper_bgcolor=PAPER_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR),
    title=dict(
        text="<b>Migrant Population, Non-Employment & Poverty Rate</b>"
             '<br><span style="font-size:13px; color:rgba(255,232,200,0.55)">'
             "Source: Eurostat  ·  Fill = foreign-citizen pop"
             "  ·  Solid = % non-employed (non-EU)"
             "  ·  Dotted = % immigrants at risk of poverty</span>",
        font=dict(color=TITLE_COLOR, size=19,
                  family="Inter, Segoe UI, sans-serif"),
        x=0.5, xanchor="center",
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,235,200,0.15)",
        borderwidth=1,
        font=dict(color=TEXT_COLOR, size=11,
                  family="Inter, sans-serif"),
        orientation="h",
        x=0.5, y=-0.06, xanchor="center",
    ),
    hovermode="x unified",
    margin=dict(l=65, r=65, t=110, b=70),
)

# Style x-axes
for i in range(1, 5):
    xa = f"xaxis{i}" if i > 1 else "xaxis"
    fig3.update_layout(**{xa: dict(
        gridcolor=GRID_COLOR, linecolor=LINE_COLOR,
        tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=11),
        dtick=2,
    )})

# Primary y (millions)
fig3.update_yaxes(
    gridcolor=GRID_COLOR, linecolor=LINE_COLOR,
    tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=11),
    ticksuffix="M",
    secondary_y=False,
)

# Secondary y (percentage, fixed 0–100%)
fig3.update_yaxes(
    gridcolor="rgba(255,232,200,0.06)",
    linecolor="rgba(255,232,200,0.15)",
    tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=10),
    ticksuffix="%",
    range=[0, 100],
    secondary_y=True,
)

# Style subplot titles
for ann in fig3.layout.annotations:
    ann.font = dict(color=TEXT_COLOR, size=14, family="Inter, sans-serif")

# ── Export ────────────────────────────────────────────────────────────
fig3.write_html("output/europe_migration_employment_welfare.html")
fig3.write_image("output/europe_migration_employment_welfare.png", scale=2)
print("Saved → output/europe_migration_employment_welfare.html  &  .png")

fig3.show()


Fetching population by citizenship …
  ✓ 64 rows  (2010–2025)
Fetching non-EU employment rates …
  ✓ 60 rows  (2010–2024)
Fetching at-risk-of-poverty rate for foreign citizens …
  ✓ 62 rows  (2010–2025)
Saved → output/europe_migration_employment_welfare.html  &  .png


In [5]:
# ══════════════════════════════════════════════════════════════════════
# 2×2 — Scaled view: metrics as portions OF the immigrant population
#   Lightest fill = total foreign-citizen pop (millions)
#   Medium  fill  = non-employed immigrants (nonemp% × foreign pop)
#   Darkest fill  = immigrants at risk of poverty (poverty% × foreign pop)
# ══════════════════════════════════════════════════════════════════════
from plotly.subplots import make_subplots
import plotly.graph_objects as go

FOCUS = {
    "DE": {"name": "Germany",  "color": "#EEB5A4"},
    "FR": {"name": "France",   "color": "#88232B"},
    "ES": {"name": "Spain",    "color": "#9FA975"},
    "IT": {"name": "Italy",    "color": "#7C756B"},
}
focus_codes = list(FOCUS.keys())


def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def rgba(h, alpha=1.0):
    r, g, b = hex_to_rgb(h)
    return f"rgba({r},{g},{b},{alpha})"

def shade(h, factor, alpha=1.0):
    r, g, b = hex_to_rgb(h)
    r = min(255, int(r * factor))
    g = min(255, int(g * factor))
    b = min(255, int(b * factor))
    if alpha < 1.0:
        return f"rgba({r},{g},{b},{alpha})"
    return f"rgb({r},{g},{b})"


def eurostat_flat(data, extra_dim=None):
    dims  = data["id"]
    sizes = data["size"]
    geo_labels = data["dimension"]["geo"]["category"]["label"]
    geo_index  = data["dimension"]["geo"]["category"]["index"]
    time_index = data["dimension"]["time"]["category"]["index"]
    extra_index, extra_labels = {}, {}
    if extra_dim:
        extra_index = data["dimension"][extra_dim]["category"]["index"]
        extra_labels = data["dimension"][extra_dim]["category"]["label"]
    def flat_idx(vec):
        idx = 0
        for i, di in enumerate(vec):
            s = 1
            for j in range(i + 1, len(sizes)):
                s *= sizes[j]
            idx += di * s
        return idx
    base = [0] * len(dims)
    rows = []
    ex_items = extra_index.items() if extra_dim else [("_", 0)]
    for ex_code, ei in ex_items:
        for geo_code, gi in geo_index.items():
            for year, ti in time_index.items():
                vec = base.copy()
                vec[dims.index("geo")]  = gi
                vec[dims.index("time")] = ti
                if extra_dim:
                    vec[dims.index(extra_dim)] = ei
                val = data["value"].get(str(flat_idx(vec)))
                if val is not None:
                    row = {"country": geo_labels[geo_code],
                           "iso2": geo_code, "year": int(year), "value": val}
                    if extra_dim:
                        row["dim_code"] = ex_code
                    rows.append(row)
    return pd.DataFrame(rows)


# ── 1. Foreign population ────────────────────────────────────────────
print("Fetching population …")
pop_resp = requests.get(
    f"{EUROSTAT_BASE}/migr_pop1ctz",
    params={"format": "JSON", "lang": "en", "sinceTimePeriod": "2010",
            "age": "TOTAL", "sex": "T", "citizen": ["TOTAL", "NAT"],
            "geo": focus_codes},
    timeout=30)
pop_resp.raise_for_status()
df_pop_raw = eurostat_flat(pop_resp.json(), extra_dim="citizen")
df_tot = df_pop_raw[df_pop_raw.dim_code == "TOTAL"][["iso2","country","year","value"]].rename(columns={"value":"pop_total"})
df_nat = df_pop_raw[df_pop_raw.dim_code == "NAT"][["iso2","year","value"]].rename(columns={"value":"pop_nat"})
df_pop = df_tot.merge(df_nat, on=["iso2","year"], how="inner")
df_pop["foreign_m"] = (df_pop.pop_total - df_pop.pop_nat) / 1e6
print(f"  ✓ {len(df_pop)} rows")


# ── 2. Non-employment rate (non-EU) ──────────────────────────────────
print("Fetching employment …")
emp_resp = requests.get(
    f"{EUROSTAT_BASE}/lfsa_ergan",
    params={"format": "JSON", "lang": "en", "sinceTimePeriod": "2010",
            "sex": "T", "age": "Y20-64", "citizen": "NEU27_2020_FOR",
            "geo": focus_codes},
    timeout=30)
emp_resp.raise_for_status()
df_emp = eurostat_flat(emp_resp.json()).rename(columns={"value": "emp_pct"})
df_emp["nonemp_pct"] = 100.0 - df_emp["emp_pct"]
print(f"  ✓ {len(df_emp)} rows")


# ── 3. Poverty rate for foreign citizens ─────────────────────────────
print("Fetching poverty …")
pov_resp = requests.get(
    f"{EUROSTAT_BASE}/ilc_li31",
    params={"format": "JSON", "lang": "en", "sinceTimePeriod": "2010",
            "sex": "T", "age": "Y_GE18", "citizen": "FOR",
            "geo": focus_codes},
    timeout=30)
pov_resp.raise_for_status()
df_pov = eurostat_flat(pov_resp.json()).rename(columns={"value": "poverty_pct"})
print(f"  ✓ {len(df_pov)} rows")


# ── Merge & compute absolute numbers ─────────────────────────────────
df_m = df_pop[["iso2","country","year","foreign_m"]].copy()
df_m = df_m.merge(df_emp[["iso2","year","nonemp_pct"]], on=["iso2","year"], how="left")
df_m = df_m.merge(df_pov[["iso2","year","poverty_pct"]], on=["iso2","year"], how="left")
df_m["nonemp_m"]  = df_m["foreign_m"] * df_m["nonemp_pct"]  / 100
df_m["poverty_m"] = df_m["foreign_m"] * df_m["poverty_pct"] / 100
print(f"  ✓ Merged: {len(df_m)} rows")


# ══════════════════════════════════════════════════════════════════════
# Build 2×2 subplots — single y-axis (millions)
# ══════════════════════════════════════════════════════════════════════
fig4 = make_subplots(
    rows=2, cols=2,
    shared_xaxes=True,
    vertical_spacing=0.10,
    horizontal_spacing=0.12,
    subplot_titles=[FOCUS[c]["name"] for c in focus_codes],
    specs=[[{"secondary_y": True}, {"secondary_y": True}],
           [{"secondary_y": True}, {"secondary_y": True}]],
)

for idx, iso2 in enumerate(focus_codes):
    row = idx // 2 + 1
    col = idx % 2 + 1
    base = FOCUS[iso2]["color"]
    s = df_m[df_m.iso2 == iso2].sort_values("year").dropna(subset=["nonemp_m"])

    # Layer 1 (top, lightest): total foreign pop
    fig4.add_trace(
        go.Scatter(
            x=s["year"], y=s["foreign_m"],
            mode="lines",
            name="Total immigrant pop",
            line=dict(width=0.5, color=rgba(base, 0.7)),
            fillcolor=rgba(base, 0.25),
            fill="tozeroy",
            showlegend=(idx == 0),
            legendgroup="total",
            hovertemplate="Total: %{y:.2f}M<extra></extra>",
        ),
        row=row, col=col,
    )

    # Layer 2 (middle, medium): non-employed immigrants
    fig4.add_trace(
        go.Scatter(
            x=s["year"], y=s["nonemp_m"],
            mode="lines",
            name="Non-employed immigrants",
            line=dict(width=1.5, color=shade(base, 0.60, 0.9)),
            fillcolor=shade(base, 0.60, 0.50),
            fill="tozeroy",
            showlegend=(idx == 0),
            legendgroup="nonemp",
            hovertemplate="Non-employed: %{y:.2f}M (%{customdata:.0f}%)<extra></extra>",
            customdata=s["nonemp_pct"],
        ),
        row=row, col=col,
    )

    # Layer 3 (bottom, darkest): immigrants at risk of poverty
    fig4.add_trace(
        go.Scatter(
            x=s["year"], y=s["poverty_m"],
            mode="lines",
            name="Immigrants at risk of poverty",
            line=dict(width=1.5, color=shade(base, 0.35, 0.9)),
            fillcolor=shade(base, 0.35, 0.65),
            fill="tozeroy",
            showlegend=(idx == 0),
            legendgroup="poverty",
            hovertemplate="At-risk poverty: %{y:.2f}M (%{customdata:.0f}%)<extra></extra>",
            customdata=s["poverty_pct"],
        ),
        row=row, col=col,
    )

    # Invisible traces on secondary y — just to anchor the 0-100% axis
    fig4.add_trace(
        go.Scatter(
            x=s["year"], y=s["nonemp_pct"],
            mode="lines", line=dict(width=0, color="rgba(0,0,0,0)"),
            showlegend=False, hoverinfo="skip",
        ),
        row=row, col=col, secondary_y=True,
    )


# ── Layout ────────────────────────────────────────────────────────────
fig4.update_layout(
    height=850,
    width=1150,
    paper_bgcolor=PAPER_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR),
    title=dict(
        text="<b>Immigrant Population — Scaled Non-Employment & Poverty</b>"
             '<br><span style="font-size:13px; color:rgba(255,232,200,0.55)">'
             "Source: Eurostat  ·  Lightest = total foreign pop"
             "  ·  Medium = non-employed share  ·  Darkest = at-risk-of-poverty share</span>",
        font=dict(color=TITLE_COLOR, size=19,
                  family="Inter, Segoe UI, sans-serif"),
        x=0.5, xanchor="center",
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,235,200,0.15)",
        borderwidth=1,
        font=dict(color=TEXT_COLOR, size=11, family="Inter, sans-serif"),
        orientation="h",
        x=0.5, y=-0.06, xanchor="center",
    ),
    hovermode="x unified",
    margin=dict(l=65, r=65, t=110, b=70),
)

# X-axes
for i in range(1, 5):
    xa = f"xaxis{i}" if i > 1 else "xaxis"
    fig4.update_layout(**{xa: dict(
        gridcolor=GRID_COLOR, linecolor=LINE_COLOR,
        tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=11),
        dtick=2,
    )})

# Primary y (millions)
fig4.update_yaxes(
    gridcolor=GRID_COLOR, linecolor=LINE_COLOR,
    tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=11),
    ticksuffix="M",
    secondary_y=False,
)

# Secondary y (percentage, 0-100%)
fig4.update_yaxes(
    gridcolor="rgba(0,0,0,0)",
    linecolor=LINE_COLOR,
    tickfont=dict(color=TEXT_COLOR, family="JetBrains Mono, monospace", size=10),
    ticksuffix="%",
    range=[0, 100],
    showgrid=False,
    secondary_y=True,
)

# Subplot titles
for ann in fig4.layout.annotations:
    ann.font = dict(color=TEXT_COLOR, size=14, family="Inter, sans-serif")

# ── Export ────────────────────────────────────────────────────────────
fig4.write_html("output/europe_migration_scaled.html")
fig4.write_image("output/europe_migration_scaled.png", scale=2)
print("Saved → output/europe_migration_scaled.html  &  .png")

fig4.show()


Fetching population …
  ✓ 64 rows
Fetching employment …
  ✓ 60 rows
Fetching poverty …
  ✓ 62 rows
  ✓ Merged: 64 rows
Saved → output/europe_migration_scaled.html  &  .png
